# Datamart Acquiring v3 — инструкция для коллеги

## Что делает тетрадка
- Собирает витрину эквайринга за выбранный месяц из Озера.
- Сверяет метрики с Excel за тот же месяц.
- Считает коэффициенты по каждому `agr_id` для `commission_monthly` (база для прогноза на следующий месяц).

## Что нужно заполнить перед запуском
В ячейке параметров:
- `report_month` — первый день месяца расчета, например `'2026-04-01'`.
- `excel_path` — путь к Excel за тот же месяц, например `'/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'`.
- `excel_header` — обычно `0`.
- `output_csv_path` — путь сохранения витрины CSV.
- `run_invalidate_metadata`:
  - `False` для обычного запуска (быстрее),
  - `True` если знаешь, что метаданные таблиц недавно обновлялись.
- `save_csv`:
  - `True` чтобы сохранить итоговую витрину,
  - `False` если нужен только расчет в памяти.

## Как запускать
1. Проверить параметры месяца и пути в ячейке параметров.
2. Запустить все ячейки сверху вниз.
3. Проверить итоговые таблицы сверки Lake vs Excel.
4. Забрать файлы:
   - витрина: `output_csv_path`,
   - коэффициенты: `commission_monthly_coef_by_agr_YYYY_MM.csv`.

## Важные правила расчета
- Периметр договоров: SA + активные terms (`cf_ter_type='P'`), без `RSHB` в периметре.
- Транзакции: фильтр `RSHB` сохранен в транзакционном шаге.
- AUR считается как `retl_cnt * 1926`.

In [ ]:
import pandas as pd

attributes_dict_df = pd.DataFrame([
    {'attribute': 'report_month', 'description_ru': 'Отчетный месяц витрины (первый день месяца)'},
    {'attribute': 'snapshot_month_start', 'description_ru': 'Техническая метка среза для BI-фильтрации'},
    {'attribute': 'inn', 'description_ru': 'ИНН клиента'},
    {'attribute': 'company_name', 'description_ru': 'Наименование клиента'},
    {'attribute': 'agr_id', 'description_ru': 'ID договора из Alpha (если есть)'},
    {'attribute': 'n_agr', 'description_ru': 'Внутренний номер договора в Alpha'},
    {'attribute': 'contract_number', 'description_ru': 'Номер договора в Alpha'},
    {'attribute': 'd_valid_from', 'description_ru': 'Дата начала действия договора'},
    {'attribute': 'd_valid_to', 'description_ru': 'Дата окончания действия договора'},
    {'attribute': 'cdi_id', 'description_ru': 'Идентификатор клиента в CDI'},
    {'attribute': 'ssp_ocrm', 'description_ru': 'Сегмент/блок ОКРМ'},
    {'attribute': 'ogrn', 'description_ru': 'ОГРН клиента'},
    {'attribute': 'filial_rf', 'description_ru': 'Филиал РФ'},
    {'attribute': 'vsp_name', 'description_ru': 'Наименование ВСП'},
    {'attribute': 'vsp_code', 'description_ru': 'Код ВСП'},
    {'attribute': 'tariff_name', 'description_ru': 'Тарифный план'},
    {'attribute': 'retl_cnt', 'description_ru': 'Количество торговых точек'},
    {'attribute': 'term_cnt', 'description_ru': 'Количество терминалов'},
    {'attribute': 'active_term_cnt', 'description_ru': 'Количество активных терминалов (>=1 trx с суммой > 1 за месяц)'},
    {'attribute': 'trx_cnt', 'description_ru': 'Количество операций за месяц'},
    {'attribute': 'trx_sum', 'description_ru': 'Сумма операций за месяц'},
    {'attribute': 'commission_from_ops', 'description_ru': 'Комиссия с операций'},
    {'attribute': 'commission_monthly', 'description_ru': 'Фиксированная комиссия в месяц'},
    {'attribute': 'int_component', 'description_ru': 'Interchange / внутренняя составляющая'},
    {'attribute': 'commission_total', 'description_ru': 'Итоговая комиссия'},
    {'attribute': 'aur', 'description_ru': 'АУР (расчет по терминалам)'},
    {'attribute': 'amortization', 'description_ru': 'Амортизация'},
    {'attribute': 'chod', 'description_ru': 'ЧОД'},
    {'attribute': 'fin_result', 'description_ru': 'Финансовый результат'},
])

# Справочник оставлен для документации, вывод отключен для компактного запуска.
print(f'Справочник атрибутов загружен: {len(attributes_dict_df)} полей')

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

# Убираем scientific notation в выводе DataFrame (например 1.6543e+08)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

In [ ]:
# Единственный параметр месяца (первый день месяца)
report_month = '2026-04-01'

# Путь для итогового CSV
output_csv_path = './datamart_month_acquiring_2026_04.csv'

# Excel для сравнения атрибутов за апрель 2026
excel_path = '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'
excel_header = 0

# Целевая доля agr_id, которые должны приходить напрямую из SA (без fallback)
target_agr_id_direct_pct = 99.0

# Управление служебными шагами
run_invalidate_metadata = False
save_csv = True
run_excel_compare = True
show_compare_top = False
save_coef_csv = True

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')
snapshot_month_start = month_start

invalidate_tables = [
    'ods_alpha.scd1_agreements',
    'ods_alpha.scd1_companies',
    'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext',
    'cdiul.ext_id_org',
    'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals',
    'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx',
    'ods_alpha.scd1_trx_acq',
    'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids',
    'ods.scd1_z_r2_ip_merchants',
    'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix',
    'ods.scd1_z_cl_corp',
    'ods.scd1_z_depart',
    'ods.scd1_z_branch',
    'ods.scd1_z_r2_tariff_plan'
]

if run_invalidate_metadata:
    invalidate_ok = 0
    invalidate_failed = []
    with imp:
        for t in invalidate_tables:
            try:
                imp.execute(f'invalidate metadata {t}')
                invalidate_ok += 1
            except Exception as e:
                invalidate_failed.append((t, str(e)))

    print(f'Invalidate успешно: {invalidate_ok}/{len(invalidate_tables)}')
    if invalidate_failed:
        print(f'Invalidate с ошибкой: {len(invalidate_failed)} (продолжаем выполнение)')
        display(pd.DataFrame(invalidate_failed, columns=['table_name', 'error_message']).head(20))
else:
    print('Invalidate metadata пропущен (run_invalidate_metadata=False) для ускорения запуска.')

In [ ]:
def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None

## SA-периметр (изменен)

Берем SA-договоры, активные в выбранном месяце, и оставляем только те, у которых существует активная запись в `ods_alpha.scd1_agr_terms` с `cf_ter_type='P'`.

Важно: здесь **нет** фильтра по `c_fiid_grp='RSHB'`.

In [ ]:
sql_sa_perimeter = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_df = imp.fetch(sql_sa_perimeter)

if sa_df is None:
    sa_df = pd.DataFrame()

if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

sa_rows_cnt = len(sa_df)
sa_inn_unique_cnt = sa_df['inn'].dropna().astype(str).nunique() if 'inn' in sa_df.columns else 0
print(f'Записей в SA-периметре (terms P, no RSHB): {sa_rows_cnt:,}')
print(f'Уникальных ИНН в SA-периметре: {sa_inn_unique_cnt:,}')

sa_df[['n_agr', 'inn', 'company_name']].head(3)

In [ ]:
inn_values = sorted([x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_cdi = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc,
               cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm, x_area_resp_norm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  o.x_area_resp_norm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cdi_map_df = imp.fetch(sql_cdi)

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])

if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
    allowed_ssp_values = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm'].where(cdi_map_df['ssp_ocrm'].isin(allowed_ssp_values), None)
    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

print(f'CDI map строк: {len(cdi_map_df):,}')
cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']].head(3)

In [ ]:
cdi_values = sorted([x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_cft = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in ({cdi_sql_list})
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_map_df = imp.fetch(sql_cft)

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])

if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'CFT map строк: {len(cft_map_df):,}')
cft_map_df.head(3)

In [ ]:
# 1) Agreement-level metrics with active terminals in month (оптимизировано)
if sa_df.empty:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
else:
    n_agrs_cmp = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in_cmp = ', '.join([f"'{x}'" for x in n_agrs_cmp]) if n_agrs_cmp else "''"

    sql_cmp = f"""
    with sa_agr as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({agr_in_cmp})
    ),
    m as (
      select
        sa.n_agr,
        sa.n_cmp_client,
        cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null
        and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    ),
    term_active as (
      select
        m.n_agr,
        m.n_cmp_client,
        m.c_nmrc,
        cast(t.c_nter as string) as c_nter
      from m
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = m.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string)
    ),
    retl as (
      select n_agr, count(distinct c_nmrc) as retl_cnt
      from term_active
      group by n_agr
    ),
    term as (
      select n_agr, count(distinct c_nter) as term_cnt
      from term_active
      group by n_agr
    ),
    amort as (
      select
        ta.n_agr,
        sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
      from term_active ta
      left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
        on cast(am.c_nter as string) = ta.c_nter
      group by ta.n_agr
    )
    select
      sa.n_agr,
      sa.n_cmp_client,
      r.retl_cnt,
      t.term_cnt,
      a.amortization
    from sa_agr sa
    left join retl r on r.n_agr = sa.n_agr
    left join term t on t.n_agr = sa.n_agr
    left join amort a on a.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        cmp_df = imp.fetch(sql_cmp)

    if cmp_df is None:
        cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])

    if not cmp_df.empty:
        cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
        cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)

    # 2) TRX metrics by agreement (RSHB в trx оставлен как в базовой тетрадке)
    n_agrs = n_agrs_cmp
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    sql_trx = f"""
    with sa_agr as (
      select distinct cast(n_agr as string) as n_agr, cast(n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements
      where cast(n_agr as string) in ({agr_in})
    ),
    trx_base_raw as (
      select
        cast(t.n_trx as string) as n_trx,
        cast(t.c_nter as string) as c_nter,
        cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      left join ods_alpha.scd1_base24_fiids fa
        on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
        and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base as (
      select
        n_trx,
        max(c_nter) as c_nter,
        max(n_amt_src) as n_amt_src
      from trx_base_raw
      group by n_trx
    ),
    ta_raw as (
      select
        cast(a.n_trx as string) as n_trx,
        cast(a.n_agr as string) as n_agr,
        coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb
        on tb.n_trx = cast(a.n_trx as string)
      where cast(a.n_agr as string) in ({agr_in})
    ),
    ta as (
      select
        n_trx,
        n_agr,
        max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_int_agg as (
      select
        cast(i.n_trx as string) as n_trx,
        sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys k
        on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    ),
    tj as (
      select ta.n_agr, sa.n_cmp_client, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
      from ta
      join trx_base tb
        on tb.n_trx = ta.n_trx
      left join sa_agr sa
        on sa.n_agr = ta.n_agr
    ),
    trx_agg as (
      select
        tj.n_agr,
        count(distinct tj.n_trx) as trx_cnt,
        sum(tj.n_amt_src) as trx_sum,
        sum(tj.n_amt_tax) as commission_from_ops,
        sum(coalesce(i.n_amt_fee, 0.0)) as int_component
      from tj
      left join trx_int_agg i
        on i.n_trx = tj.n_trx
      group by tj.n_agr
    ),
    active_term_agg as (
      select
        tj.n_cmp_client,
        count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
      from tj
      where tj.n_cmp_client is not null
      group by tj.n_cmp_client
    )
    select
      t.n_agr,
      t.trx_cnt,
      t.trx_sum,
      t.commission_from_ops,
      t.int_component,
      a.n_cmp_client,
      a.active_term_cnt
    from trx_agg t
    left join sa_agr sa
      on sa.n_agr = t.n_agr
    left join active_term_agg a
      on a.n_cmp_client = sa.n_cmp_client
    """

    with imp:
        imp.execute('set MEM_LIMIT=16g')
        trx_df = imp.fetch(sql_trx)

    if trx_df is None:
        trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])

    if not trx_df.empty:
        trx_df['n_agr'] = trx_df['n_agr'].astype(str)
        trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)

active_term_df = (
    trx_df[['n_cmp_client', 'active_term_cnt']]
    .dropna(subset=['n_cmp_client'])
    .drop_duplicates(subset=['n_cmp_client'])
    if len(trx_df)
    else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
)

# 3) R2 attributes вынесены в следующую секцию

# 4) Final assembly вынесена в отдельную секцию

## Секция 4. R2 атрибуты
Кратко: получаем оргструктуру и фикс-комиссию по `cft_id` для обогащения витрины.

In [ ]:
# R2 атрибуты: оргструктура, тариф и фикс-комиссия
cft_values = sorted([x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

if not cft_values:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name', 'commission_monthly'])
else:
    sql_r2 = f"""
    with r2 as (
      select
        cast(m.id as string) as r2_id,
        cast(m.c_cl_org as string) as cft_id,
        cast(m.c_depart as string) as c_depart,
        cast(m.c_tariff_plan as string) as c_tariff_plan
      from ods.scd1_z_r2_ip_merchants m
      where cast(m.c_cl_org as string) in ({cft_sql_list})
    ),
    fix as (
      select cast(tt.c_tariff_plan as string) as c_tariff_plan, max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_tune tt
      left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
      group by cast(tt.c_tariff_plan as string)
    )
    select
      r2.r2_id,
      r2.cft_id,
      cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
      cast(dep.c_name as string) as vsp_name,
      cast(dep.c_num as string) as vsp_code,
      case
        when br.c_shortlabel is null then null
        when upper(cast(br.c_shortlabel as string)) like '%РФ%'
          then regexp_extract(cast(br.c_shortlabel as string), '^(.*?РФ)', 1)
        else cast(br.c_shortlabel as string)
      end as filial_rf,
      cast(tp.c_name as string) as tariff_name,
      cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly
    from r2
    left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = r2.cft_id
    left join ods.scd1_z_depart dep on cast(dep.id as string) = r2.c_depart
    left join ods.scd1_z_branch br on cast(br.id as string) = cast(dep.c_filial as string)
    left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = r2.c_tariff_plan
    left join fix on fix.c_tariff_plan = r2.c_tariff_plan
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        r2_df = imp.fetch(sql_r2)

    if r2_df is None:
        r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name', 'commission_monthly'])

if not r2_df.empty:
    for c in ['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name']:
        r2_df[c] = r2_df[c].astype(str)
    r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')

print(f'R2 rows: {len(r2_df):,}')

## Секция 5. Сборка финальной витрины
Кратко: объединяем SA, CDI/CFT, операционные и транзакционные метрики, считаем финальные показатели и формируем `final_df`.

In [ ]:
# Final assembly + fallback по agr_id + финальные формулы
base_df = sa_df.copy()

if not cdi_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']], on='inn', how='left')
else:
    base_df['cdi_id'] = None
    base_df['ssp_ocrm'] = None

if not cft_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
else:
    for col in ['r2_id', 'commission_monthly', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name']:
        base_df[col] = None

if not cmp_df.empty and not base_df.empty:
    base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
else:
    for col in ['retl_cnt', 'term_cnt', 'amortization']:
        base_df[col] = None

if 'active_term_df' in globals() and not active_term_df.empty and not base_df.empty:
    base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
else:
    base_df['active_term_cnt'] = None

if not trx_df.empty and not base_df.empty:
    base_df = base_df.merge(trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']], on='n_agr', how='left')
else:
    for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']:
        base_df[col] = None

if 'agr_id' not in base_df.columns:
    base_df['agr_id'] = None
if 'r2_id' not in base_df.columns:
    base_df['r2_id'] = None

agr_id_before_fill = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
base_df['agr_id_source'] = 'sa'
fallback_mask = (~agr_id_before_fill) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

agr_id_after_fill = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
missing_after_fill_mask = ~agr_id_after_fill

total_rows = int(len(base_df))
sa_direct_rows = int(agr_id_before_fill.sum())
fallback_rows = int(fallback_mask.sum())
missing_after_rows = int(missing_after_fill_mask.sum())

sa_direct_pct = round(100.0 * sa_direct_rows / total_rows, 2) if total_rows else 0.0
fallback_pct = round(100.0 * fallback_rows / total_rows, 2) if total_rows else 0.0
missing_after_pct = round(100.0 * missing_after_rows / total_rows, 2) if total_rows else 0.0
agr_filled_after_pct = round(100.0 * agr_id_after_fill.mean(), 2) if total_rows else 0.0

agr_source_stats_df = pd.DataFrame([
    {'source': 'sa_direct', 'rows_cnt': sa_direct_rows, 'rows_pct': sa_direct_pct},
    {'source': 'r2_fallback', 'rows_cnt': fallback_rows, 'rows_pct': fallback_pct},
    {'source': 'missing_after_fallback', 'rows_cnt': missing_after_rows, 'rows_pct': missing_after_pct},
])

print(f'Всего строк: {total_rows:,}')
print(f'agr_id из SA (без fallback): {sa_direct_rows:,} ({sa_direct_pct}%)')
print(f'agr_id через fallback (r2_id): {fallback_rows:,} ({fallback_pct}%)')
print(f'agr_id заполнено после fallback: {agr_filled_after_pct}%')
print(f'agr_id отсутствует после fallback: {missing_after_rows:,} ({missing_after_pct}%)')

if sa_direct_pct >= target_agr_id_direct_pct:
    print(f'OK: доля agr_id из SA >= {target_agr_id_direct_pct}%')
else:
    print(f'WARN: доля agr_id из SA ниже целевого порога {target_agr_id_direct_pct}%')

display(agr_source_stats_df)

commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)

base_df['commission_total'] = commission_from_ops_num + commission_monthly_num

# AUR по бизнес-правилу: только от количества торговых точек
retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)
base_df['aur'] = retl_cnt_num * 1926

aur_num = pd.to_numeric(base_df.get('aur'), errors='coerce').fillna(0)
base_df['chod'] = base_df['commission_total'] + int_component_num
base_df['fin_result'] = base_df['chod'] - aur_num - amortization_num
base_df['report_month'] = report_month_label
base_df['snapshot_month_start'] = snapshot_month_start

mvp_columns = [
    'report_month', 'snapshot_month_start', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'cdi_id', 'ssp_ocrm', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name',
    'retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    if col in base_df.columns:
        base_df[col] = base_df[col].map(to_decimal_or_none)

for col in ['retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt']:
    if col in base_df.columns:
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df = base_df[mvp_columns].copy()

print(f'Собрано строк: {len(final_df):,}')
print(f'Атрибутов в финале: {len(final_df.columns)}')

## Секция 6. Экспорт витрины
Кратко: сохраняем `final_df` в CSV, если включен флаг `save_csv`.

In [ ]:
if save_csv:
    output_path = Path(output_csv_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'CSV сохранен: {output_path.resolve()}')
else:
    print('Экспорт в CSV отключен (save_csv=False). Результат доступен в DataFrame `final_df`.')

print(f'Строк: {len(final_df):,}')

In [ ]:
# Блок TOP-10 по условию бизнес-ошибки перенесен в Секцию 8.
# Оставлено пустым специально, чтобы не было раннего падения при Run All.


## Секция 7. Сверка с Excel и коэффициенты
Кратко: сравниваем Lake vs Excel по ключу `inn+agr_id`, считаем KPI и формируем коэффициенты `commission_monthly` по каждому `agr_id`.

In [ ]:
# Нормализация ключей для финальной сверки с Excel

def normalize_inn_q1(value):
    if pd.isna(value):
        return None

    s = str(value).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None

    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)

    if len(s) not in (10, 12):
        return None
    return s


def normalize_agr_q1(value):
    if pd.isna(value):
        return None

    s = str(value).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None

    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass

    s = re.sub(r'\.0$', '', s)
    if s in {'', 'nan', 'None'}:
        return None
    return s

In [ ]:
row_cnt = len(final_df)
field_fill_stats = []
for col in final_df.columns:
    non_null_cnt = int(final_df[col].notna().sum())
    null_cnt = int(row_cnt - non_null_cnt)
    fill_rate_pct = round((non_null_cnt / row_cnt) * 100, 2) if row_cnt else 0.0
    field_fill_stats.append({
        'field_name': col,
        'non_null_cnt': non_null_cnt,
        'null_cnt': null_cnt,
        'fill_rate_pct': fill_rate_pct
    })

fill_stats_df = pd.DataFrame(field_fill_stats).sort_values(['fill_rate_pct', 'field_name'], ascending=[True, True]).reset_index(drop=True)
print(f'Строк в финальной витрине: {row_cnt:,}')
fill_stats_df.head(20)

In [ ]:
final_df[['report_month', 'inn', 'agr_id', 'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum', 'commission_monthly']].head(3)

In [ ]:
# Дублирующий блок экспорта удален: используйте секцию 6 выше.

In [ ]:
# Финальная сверка расхождений Lake vs Excel (по пересечению inn+agr_id)
# + дополнительные метрики: комиссия и ЧОД

def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def exact_match_rate_from_abs(abs_series, tol=0.0):
    if len(abs_series) == 0:
        return 0.0
    return round((abs_series.fillna(0) <= tol).mean() * 100, 2)


if run_excel_compare:
    if 'excel_raw_m' in globals():
        ex = excel_raw_m.copy()
    else:
        ex = pd.read_excel(excel_path, header=excel_header)

    col_map = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
        'term_col': ['Кол-во терминалов', 'Количество терминалов'],
        'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
        'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
        'comm_ops_col': ['Комиссия (% с операций)', 'Комиссия \n(% с операций)', 'Комиссия % с операций'],
        'comm_monthly_col': ['Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)', 'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)'],
        'comm_total_col': ['Комиссия эквайринга'],
        'irf_col': ['Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)', 'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)'],
        'chod_col': ['ЧОД'],
    }

    resolved = {k: pick_col_robust(ex.columns, v) for k, v in col_map.items()}
    missing = [k for k, v in resolved.items() if v is None]
    if missing:
        raise ValueError(f'Не найдены колонки Excel: {missing}. Доступные: {list(ex.columns)}')

    print('Используемые колонки Excel:')
    display(pd.DataFrame([{'logical_name': k, 'excel_column': v} for k, v in resolved.items()]))

    ex['inn_key'] = ex[resolved['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved['agr_col']].apply(normalize_agr_q1)

    ex['retl_cnt_excel'] = pd.to_numeric(ex[resolved['retl_col']], errors='coerce')
    ex['term_cnt_excel'] = pd.to_numeric(ex[resolved['term_col']], errors='coerce')
    ex['trx_cnt_excel'] = pd.to_numeric(ex[resolved['trx_cnt_col']], errors='coerce')
    ex['trx_sum_excel'] = to_num_series(ex[resolved['trx_sum_col']])

    ex['commission_from_ops_excel'] = to_num_series(ex[resolved['comm_ops_col']])
    ex['commission_monthly_excel'] = to_num_series(ex[resolved['comm_monthly_col']])
    ex['commission_total_excel'] = to_num_series(ex[resolved['comm_total_col']])
    ex['int_component_excel'] = to_num_series(ex[resolved['irf_col']])
    ex['chod_excel'] = to_num_series(ex[resolved['chod_col']])

    ex_agg = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_excel': 'max',
              'term_cnt_excel': 'max',
              'trx_cnt_excel': 'max',
              'trx_sum_excel': 'sum',
              'commission_from_ops_excel': 'sum',
              'commission_monthly_excel': 'max',
              'commission_total_excel': 'sum',
              'int_component_excel': 'sum',
              'chod_excel': 'sum',
          })
    )

    lk = final_df.copy()
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    lk['retl_cnt_lake'] = pd.to_numeric(lk['retl_cnt'], errors='coerce')
    lk['term_cnt_lake'] = pd.to_numeric(lk['term_cnt'], errors='coerce')
    lk['trx_cnt_lake'] = pd.to_numeric(lk['trx_cnt'], errors='coerce')
    lk['trx_sum_lake'] = pd.to_numeric(lk['trx_sum'], errors='coerce')
    lk['commission_from_ops_lake'] = pd.to_numeric(lk['commission_from_ops'], errors='coerce')
    lk['commission_monthly_lake'] = pd.to_numeric(lk['commission_monthly'], errors='coerce')
    lk['commission_total_lake'] = pd.to_numeric(lk['commission_total'], errors='coerce')
    lk['int_component_lake'] = pd.to_numeric(lk['int_component'], errors='coerce')
    lk['chod_lake'] = pd.to_numeric(lk['chod'], errors='coerce')

    lake_dups = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .size()
          .query('size > 1')
          .sort_values('size', ascending=False)
    )
    print(f'Ключей inn+agr_id с дублями в Lake: {len(lake_dups)}')
    if len(lake_dups):
        display(lake_dups.head(10))

    lk_agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg({
              'retl_cnt_lake': 'max',
              'term_cnt_lake': 'max',
              'trx_cnt_lake': 'max',
              'trx_sum_lake': 'max',
              'commission_from_ops_lake': 'max',
              'commission_monthly_lake': 'max',
              'commission_total_lake': 'max',
              'int_component_lake': 'max',
              'chod_lake': 'max',
          })
    )

    cmp = lk_agg.merge(ex_agg, on=['inn_key', 'agr_id_key'], how='inner')

    metric_keys = [
        'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
        'commission_from_ops', 'commission_monthly', 'commission_total', 'int_component', 'chod'
    ]

    for m in metric_keys:
        cmp[f'{m}_delta'] = cmp[f'{m}_lake'].fillna(0) - cmp[f'{m}_excel'].fillna(0)
        cmp[f'{m}_abs'] = cmp[f'{m}_delta'].abs()


def show_top(metric_key, title):
    cols = ['agr_id_key', 'inn_key', f'{metric_key}_lake', f'{metric_key}_excel', f'{metric_key}_delta']
    print(title)
    display(cmp.sort_values(f'{metric_key}_abs', ascending=False)[cols].head(10))


if show_compare_top:
    show_top('retl_cnt', 'TOP-10 retl_cnt')
    show_top('term_cnt', 'TOP-10 term_cnt')
    show_top('trx_cnt', 'TOP-10 trx_cnt')
    show_top('trx_sum', 'TOP-10 trx_sum')
    show_top('commission_from_ops', 'TOP-10 Комиссия (% с операций)')
    show_top('commission_monthly', 'TOP-10 Комиссия (₽ в месяц)')
    show_top('commission_total', 'TOP-10 Комиссия эквайринга')
    show_top('int_component', 'TOP-10 Комиссия МПС (IRF, ₽)')
    show_top('chod', 'TOP-10 ЧОД')

    count_metrics = ['retl_cnt', 'term_cnt', 'trx_cnt']
    money_metrics = ['trx_sum', 'commission_from_ops', 'commission_monthly', 'commission_total', 'int_component', 'chod']

    kpi_rows = [{'metric': 'rows_compared', 'value': len(cmp)}]
    for m in count_metrics:
        kpi_rows.append({'metric': f'{m}_exact_match_rate_pct', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.0)})
        kpi_rows.append({'metric': f'{m}_mae', 'value': float(cmp[f'{m}_abs'].mean()) if len(cmp) else 0.0})
    for m in money_metrics:
        kpi_rows.append({'metric': f'{m}_exact_match_rate_pct', 'value': exact_match_rate_from_abs(cmp[f'{m}_abs'], tol=0.01)})
        kpi_rows.append({'metric': f'{m}_mae', 'value': float(cmp[f'{m}_abs'].mean()) if len(cmp) else 0.0})

    kpi_compare_df = pd.DataFrame(kpi_rows)
    print(f'KPI расхождений Lake vs Excel ({report_month_label}):')
    display(kpi_compare_df)

    totals_row = {'report_month': report_month, 'rows_compared': len(cmp)}
    for m in metric_keys:
        totals_row[f'excel_{m}_total'] = cmp[f'{m}_excel'].fillna(0).sum()
        totals_row[f'lake_{m}_total'] = cmp[f'{m}_lake'].fillna(0).sum()
        totals_row[f'delta_{m}_total'] = totals_row[f'lake_{m}_total'] - totals_row[f'excel_{m}_total']

    totals = pd.DataFrame([totals_row])
    print(f'Итоги за месяц {report_month} (пересечение inn+agr_id):')
    display(totals)

    # Коэффициенты для commission_monthly по каждому agr_id (для прогноза следующего месяца)
    coef_by_agr = (
        cmp[['agr_id_key', 'inn_key', 'commission_monthly_lake', 'commission_monthly_excel']]
          .groupby('agr_id_key', as_index=False)
          .agg(
              inn_key=('inn_key', 'first'),
              commission_monthly_lake_base=('commission_monthly_lake', 'sum'),
              commission_monthly_excel_base=('commission_monthly_excel', 'sum')
          )
    )

    coef_by_agr['k_comm_monthly_agr'] = coef_by_agr['commission_monthly_excel_base'] / coef_by_agr['commission_monthly_lake_base'].replace({0: pd.NA})
    coef_by_agr['k_comm_monthly_agr'] = pd.to_numeric(coef_by_agr['k_comm_monthly_agr'], errors='coerce')
    coef_by_agr['coef_source'] = 'agr_ratio'

    median_k = float(coef_by_agr['k_comm_monthly_agr'].dropna().median()) if coef_by_agr['k_comm_monthly_agr'].notna().any() else 1.0

    both_zero_mask = (coef_by_agr['commission_monthly_lake_base'] == 0) & (coef_by_agr['commission_monthly_excel_base'] == 0)
    coef_by_agr.loc[both_zero_mask, 'k_comm_monthly_agr'] = 1.0
    coef_by_agr.loc[both_zero_mask, 'coef_source'] = 'both_zero_eq_1'

    fallback_mask = coef_by_agr['k_comm_monthly_agr'].isna()
    coef_by_agr.loc[fallback_mask, 'k_comm_monthly_agr'] = median_k
    coef_by_agr.loc[fallback_mask, 'coef_source'] = 'fallback_global_median'

    coef_by_agr = coef_by_agr.sort_values('agr_id_key').reset_index(drop=True)

    print('Коэффициенты commission_monthly по agr_id (база: текущий месяц):')
    print(f'agr_id коэффициентов: {len(coef_by_agr):,}; median fallback k={median_k:,.6f}')
    display(coef_by_agr.head(20))

    if save_coef_csv:
        coef_path = Path(f'./commission_monthly_coef_by_agr_{report_month_label.replace("-", "_")}.csv')
        coef_by_agr.to_csv(coef_path, index=False, encoding='utf-8-sig')
        print(f'CSV коэффициентов сохранен: {coef_path.resolve()}')
else:
    print('Сверка с Excel и расчет коэффициентов пропущены (run_excel_compare=False).')

## Секция 8. Упрощенный отбор agr_id и выгрузка транзакций
Кратко: отбираем до 3 `agr_id` с разными тарифами, где `commission_monthly` в Excel больше, чем в Озере; затем выгружаем все транзакции за апрель в Excel (отдельный лист на каждый `agr_id`).

In [ ]:
# Упрощенный отбор: до 3 agr_id с разными тарифами
# Условие: commission_monthly в Excel больше, чем в Озере
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните секцию сверки с Excel (где формируется DataFrame cmp).')

if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните секцию сборки финальной витрины (где формируется DataFrame final_df).')

cmp_cases = cmp.copy()

# enrich cmp -> tariff_name / n_agr
lk_keys = final_df.copy()
lk_keys['inn_key'] = lk_keys['inn'].apply(normalize_inn_q1)
lk_keys['agr_id_key'] = lk_keys['agr_id'].apply(normalize_agr_q1)
lk_keys = lk_keys[['inn_key', 'agr_id_key', 'tariff_name', 'n_agr']].drop_duplicates(['inn_key', 'agr_id_key'])
cmp_cases = cmp_cases.merge(lk_keys, on=['inn_key', 'agr_id_key'], how='left', suffixes=('', '_final'))

# В cmp дельта считается как Lake - Excel. Нужно Excel > Lake => delta < 0.
simple_cand = cmp_cases[pd.to_numeric(cmp_cases['commission_monthly_delta'], errors='coerce').fillna(0) < 0].copy()
simple_cand['tariff_name'] = simple_cand['tariff_name'].astype(str).str.strip()
simple_cand['commission_monthly_gap_abs'] = pd.to_numeric(simple_cand['commission_monthly_delta'], errors='coerce').abs()

# Явно исключаем agr_id из анализа (по запросу)
excluded_agr_ids = {'1146116692539'}
simple_cand = simple_cand[~simple_cand['agr_id_key'].astype(str).isin(excluded_agr_ids)].copy()

# Фиксированные тарифные группы: Индивидуальный / Стандарт / Гибкий чек
simple_cand['tariff_name_norm'] = simple_cand['tariff_name'].astype(str).str.lower()

ind_top = simple_cand[simple_cand['tariff_name_norm'].str.contains('индивиду', na=False)].sort_values('commission_monthly_gap_abs', ascending=False).head(1)
std_top = simple_cand[simple_cand['tariff_name_norm'].str.contains('стандарт', na=False)].sort_values('commission_monthly_gap_abs', ascending=False).head(1)
flex_top = simple_cand[simple_cand['tariff_name_norm'].str.contains('гибк', na=False) & simple_cand['tariff_name_norm'].str.contains('чек', na=False)].sort_values('commission_monthly_gap_abs', ascending=False).head(1)

cases_main = pd.concat([
    ind_top.assign(case_group='individual_main'),
    std_top.assign(case_group='standard_main'),
    flex_top.assign(case_group='flex_check_main'),
], ignore_index=True)

# удалим пустые строки если какой-то тариф не найден
if 'agr_id_key' in cases_main.columns:
    cases_main = cases_main.dropna(subset=['agr_id_key']).reset_index(drop=True)

# Добавляем сравнение по метрикам операций и терминалов
for m in ['trx_cnt', 'trx_sum', 'term_cnt']:
    lake_col = f'{m}_lake'
    excel_col = f'{m}_excel'
    delta_col = f'{m}_delta'
    if lake_col in cases_main.columns and excel_col in cases_main.columns and delta_col not in cases_main.columns:
        cases_main[delta_col] = pd.to_numeric(cases_main[lake_col], errors='coerce').fillna(0) - pd.to_numeric(cases_main[excel_col], errors='coerce').fillna(0)

case_cols = [
    'agr_id_key', 'inn_key', 'n_agr', 'tariff_name',
    'term_cnt_lake', 'term_cnt_excel', 'term_cnt_delta',
    'trx_cnt_lake', 'trx_cnt_excel', 'trx_cnt_delta',
    'trx_sum_lake', 'trx_sum_excel', 'trx_sum_delta',
    'commission_monthly_lake', 'commission_monthly_excel', 'commission_monthly_delta',
    'commission_monthly_gap_abs'
]
case_cols = [c for c in case_cols if c in cases_main.columns]

print(f'Кандидатов (Excel > Lake по commission_monthly), после исключений: {len(simple_cand):,}')
print(f'Исключенные agr_id: {sorted(excluded_agr_ids)}')
print(f'Отобрано agr_id по целевым тарифам: {len(cases_main):,}')

print('Сравнение Lake vs Excel для выбранных agr_id (term_cnt, trx_cnt, trx_sum):')
display(cases_main[case_cols])

selected_agr_ids_simple = cases_main['agr_id_key'].astype(str).tolist()
print('selected_agr_ids_simple =', selected_agr_ids_simple)

# Контроль покрытия тарифов
has_ind = cases_main['tariff_name'].astype(str).str.lower().str.contains('индивиду', na=False).any() if len(cases_main) else False
has_std = cases_main['tariff_name'].astype(str).str.lower().str.contains('стандарт', na=False).any() if len(cases_main) else False
has_flex = (cases_main['tariff_name'].astype(str).str.lower().str.contains('гибк', na=False) & cases_main['tariff_name'].astype(str).str.lower().str.contains('чек', na=False)).any() if len(cases_main) else False

print(f'Покрытие тарифов: Индивидуальный={has_ind}, Стандарт={has_std}, Гибкий чек={has_flex}')
if not (has_ind and has_std and has_flex):
    print('WARN: Не все три требуемых тарифа найдены среди кандидатов.')

print('Проверка TOP-10 кандидатов по правилу бизнес-ошибки вынесена в отдельную секцию в конце тетрадки.')

In [ ]:
# Выгрузка ВСЕХ транзакций по отобранным agr_id в Excel (отдельный лист на каждый agr_id)
if 'cases_main' not in globals() or cases_main is None or len(cases_main) == 0:
    raise RuntimeError('Сначала выполните упрощенную ячейку отбора (cases_main).')

meeting_cases = cases_main.copy()
meeting_case_cards = meeting_cases[[
    c for c in [
        'agr_id_key', 'inn_key', 'n_agr', 'tariff_name',
        'trx_cnt_lake', 'trx_cnt_excel', 'trx_cnt_delta',
        'trx_sum_lake', 'trx_sum_excel', 'trx_sum_delta',
        'commission_monthly_lake', 'commission_monthly_excel', 'commission_monthly_delta',
        'commission_monthly_gap_abs'
    ] if c in meeting_cases.columns
]].copy()

print('Карточки выбранных кейсов:')
display(meeting_case_cards)

agr_ids_for_meeting = sorted(
    [str(x) for x in meeting_cases.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x]
)

if not agr_ids_for_meeting:
    print('WARN: В выбранных кейсах нет n_agr, trx-выгрузка пропущена.')
    meeting_trx_all = pd.DataFrame()
else:
    agr_in = ', '.join([f"'{x}'" for x in agr_ids_for_meeting])

    sql_meeting_trx = f"""
    with ta_raw as (
      select
        cast(a.n_trx as string) as n_trx,
        cast(a.n_agr as string) as n_agr,
        coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      where cast(a.n_agr as string) in ({agr_in})
    ),
    ta as (
      select
        n_trx,
        n_agr,
        max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_base_raw as (
      select
        cast(t.n_trx as string) as n_trx,
        cast(t.c_nter as string) as c_nter,
        cast(t.n_amt_src as double) as n_amt_src,
        cast(to_date(cast(t.d_trx_orig as timestamp)) as string) as trx_date,
        cast(t.n_mcc as string) as mcc_code,
        cast(t.c_fiid_iss as string) as c_fiid_iss,
        cast(fi_iss.c_fiid_grp as string) as c_fiid_iss_grp,
        cast(fi_acq.c_fiid_grp as string) as c_fiid_acq_grp,
        cast(fi_acq.c_fiid_desc as string) as payment_system
      from ods_alpha.scd1_trx t
      join trx_keys k
        on k.n_trx = cast(t.n_trx as string)
      left join ods_alpha.scd1_base24_fiids fi_iss
        on cast(fi_iss.c_fiid as string) = cast(t.c_fiid_iss as string)
      left join ods_alpha.scd1_base24_fiids fi_acq
        on cast(fi_acq.c_fiid as string) = cast(t.c_fiid_acq as string)
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
        and coalesce(cast(fi_acq.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base as (
      select
        n_trx,
        max(c_nter) as c_nter,
        max(n_amt_src) as n_amt_src,
        max(trx_date) as trx_date,
        max(mcc_code) as mcc_code,
        max(c_fiid_iss) as c_fiid_iss,
        max(c_fiid_iss_grp) as c_fiid_iss_grp,
        max(c_fiid_acq_grp) as c_fiid_acq_grp,
        max(payment_system) as payment_system
      from trx_base_raw
      group by n_trx
    ),
    trx_keys_valid as (
      select distinct n_trx
      from trx_base
    ),
    trx_int as (
      select
        cast(i.n_trx as string) as n_trx,
        sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys_valid k
        on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    )
    select
      ta.n_agr,
      ta.n_trx,
      tb.trx_date,
      tb.c_nter,
      tb.mcc_code,
      tb.c_fiid_iss,
      tb.c_fiid_iss_grp,
      tb.c_fiid_acq_grp,
      tb.payment_system,
      tb.n_amt_src as trx_sum_src,
      ta.n_amt_tax as commission_from_ops,
      coalesce(ti.n_amt_fee, 0.0) as int_component
    from ta
    join trx_base tb
      on tb.n_trx = ta.n_trx
    left join trx_int ti
      on ti.n_trx = ta.n_trx
    order by ta.n_agr, ta.n_trx
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        meeting_trx_all = imp.fetch(sql_meeting_trx)

    if meeting_trx_all is None:
        meeting_trx_all = pd.DataFrame()

    n_agr_to_agr_id = (
        meeting_cases[['n_agr', 'agr_id_key']]
        .dropna(subset=['n_agr', 'agr_id_key'])
        .drop_duplicates(subset=['n_agr'])
    )
    if len(meeting_trx_all) and len(n_agr_to_agr_id):
        meeting_trx_all['n_agr'] = meeting_trx_all['n_agr'].astype(str)
        n_agr_to_agr_id['n_agr'] = n_agr_to_agr_id['n_agr'].astype(str)
        meeting_trx_all = meeting_trx_all.merge(n_agr_to_agr_id, on='n_agr', how='left')

    print(f'Выгружено всех транзакций по выбранным agr_id: {len(meeting_trx_all):,}')
    display(meeting_trx_all.head(20))

    output_meeting_xlsx = Path(f'./meeting_cases_apr_trx_{report_month_label.replace("-", "_")}.xlsx')
    output_meeting_xlsx.parent.mkdir(parents=True, exist_ok=True)

    def safe_sheet_name(v):
        s = f'agr_{v}'
        # Без backslash в литерале: исключаем ошибки копипаста/экранирования
        bad_chars = '/:*?[]'
        for ch in bad_chars:
            s = s.replace(ch, '_')
        s = s.replace(chr(92), '_')  # chr(92) == '\\'
        return s[:31]

    with pd.ExcelWriter(output_meeting_xlsx, engine='xlsxwriter') as writer:
        for _, row in meeting_cases.iterrows():
            agr_key = str(row.get('agr_id_key'))
            df_sheet = meeting_trx_all[meeting_trx_all.get('agr_id_key', pd.Series(dtype=object)).astype(str) == agr_key].copy()
            if 'agr_id_key' not in df_sheet.columns:
                df_sheet = meeting_trx_all[meeting_trx_all['n_agr'].astype(str) == str(row.get('n_agr'))].copy()
            sheet_name = safe_sheet_name(agr_key)
            if df_sheet.empty:
                pd.DataFrame(columns=meeting_trx_all.columns).to_excel(writer, sheet_name=sheet_name, index=False)
            else:
                df_sheet.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f'Excel с транзакциями сохранен: {output_meeting_xlsx.resolve()}')

## Секция 9. Быстрый поиск TOP-10 кандидатов (без перезагрузки витрины)
Кратко: секция использует уже загруженные в памяти `cmp` и `final_df` и ищет 10 `agr_id` по тарифу `Стандарт`, где `commission_monthly_excel > term_cnt_lake * commission_monthly_lake`.

In [ ]:
# TOP-10 кандидатов (только Стандарт) по правилу бизнес-ошибки
# Важно: эта ячейка НЕ запускает SQL и работает на уже загруженных cmp/final_df
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните секцию сверки с Excel, где формируется DataFrame cmp.')

if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала выполните секцию сборки финальной витрины, где формируется DataFrame final_df.')

std_err = cmp.copy()

lk_ref = final_df.copy()
lk_ref['inn_key'] = lk_ref['inn'].apply(normalize_inn_q1)
lk_ref['agr_id_key'] = lk_ref['agr_id'].apply(normalize_agr_q1)
ref_cols = ['inn_key', 'agr_id_key', 'company_name', 'inn', 'tariff_name', 'n_agr']
ref_cols = [c for c in ref_cols if c in lk_ref.columns]
lk_ref = lk_ref[ref_cols].drop_duplicates(['inn_key', 'agr_id_key'])

std_err = std_err.merge(
    lk_ref,
    on=['inn_key', 'agr_id_key'],
    how='left',
    suffixes=('', '_from_final')
)

for c in ['company_name', 'inn', 'tariff_name', 'n_agr']:
    c_from = f'{c}_from_final'
    if c not in std_err.columns and c_from in std_err.columns:
        std_err[c] = std_err[c_from]
    elif c in std_err.columns and c_from in std_err.columns:
        std_err[c] = std_err[c].fillna(std_err[c_from])

for num_col in ['term_cnt_lake', 'commission_monthly_lake', 'commission_monthly_excel']:
    std_err[num_col] = pd.to_numeric(std_err.get(num_col), errors='coerce').fillna(0.0)

std_err['commission_monthly_lake_x_terms'] = std_err['term_cnt_lake'] * std_err['commission_monthly_lake']
std_err['delta_excel_vs_lake'] = std_err['commission_monthly_excel'] - std_err['commission_monthly_lake']
std_err['delta_excel_vs_lake_x_terms'] = std_err['commission_monthly_excel'] - std_err['commission_monthly_lake_x_terms']
std_err['overrun_abs'] = std_err['delta_excel_vs_lake_x_terms']

std_err['tariff_name'] = std_err.get('tariff_name', pd.Series(dtype=object)).astype(str).str.strip()
std_err['tariff_name_norm'] = std_err['tariff_name'].str.lower()

std_err = std_err[std_err['tariff_name_norm'].str.contains('стандарт', na=False)].copy()
std_err = std_err[std_err['delta_excel_vs_lake_x_terms'] > 0].copy()

excluded_agr_ids_std = set(globals().get('excluded_agr_ids', set()))
if excluded_agr_ids_std:
    std_err = std_err[~std_err['agr_id_key'].astype(str).isin(excluded_agr_ids_std)].copy()

std_err['agr_id'] = std_err['agr_id_key'].astype(str)
std_err = std_err.sort_values('overrun_abs', ascending=False)
std_error_top10 = std_err.drop_duplicates(subset=['agr_id']).head(10).copy()

cond_cnt = len(std_error_top10) <= 10
cond_formula = (std_error_top10['delta_excel_vs_lake_x_terms'] > 0).all() if len(std_error_top10) else True
cond_tariff = std_error_top10['tariff_name'].astype(str).str.lower().str.contains('стандарт', na=False).all() if len(std_error_top10) else True

print(f"Найдено кандидатов по правилу (до TOP-10): {len(std_error_top10):,}")
print(f"Контроль размера <=10: {cond_cnt}")
print(f"Контроль формулы (Excel > term_cnt*Lake): {cond_formula}")
print(f"Контроль тарифа (содержит 'стандарт'): {cond_tariff}")
if len(std_error_top10) < 10:
    print('WARN: найдено меньше 10 кандидатов, показываем всех доступных.')

std_error_cols = [
    'agr_id', 'company_name', 'inn', 'tariff_name',
    'term_cnt_lake', 'commission_monthly_lake', 'commission_monthly_excel',
    'delta_excel_vs_lake', 'delta_excel_vs_lake_x_terms'
]
std_error_cols = [c for c in std_error_cols if c in std_error_top10.columns]

print('TOP-10 кандидатов для проверки бизнесом:')
display(std_error_top10[std_error_cols])

## Секция 10. Апрель: сегментация Индивидуального тарифа на бюджет/небюджет
Кратко: для всех клиентов с тарифом `%Индивидуальный%` за апрель подтягиваем флаги `budget_cmp` и `non_budget` из `ods_alpha.scd1_companies`, строим сегмент (`budget`, `non_budget`, `conflict_both_flags`, `unknown`) и отдельно проверяем `TARGET_AGR_ID/TARGET_INN` с учетом того, что `agr_id` может быть из R2 (`r2_id`).

In [ ]:
import re
import pandas as pd

# Параметры проверки
TARGET_AGR_ID = '1182378586378'
TARGET_INN = '7701113654'
APRIL_MONTH = '2026-04-01'
CHUNK_SIZE = 1000

if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала сформируйте final_df.')
if 'imp' not in globals():
    raise RuntimeError('Нет подключения imp. Сначала выполните ячейку с connect().')

april_label = pd.to_datetime(APRIL_MONTH).strftime('%Y-%m')
if 'report_month' in globals() and str(report_month) != APRIL_MONTH:
    print(f'WARN: report_month в сессии = {report_month}, а секция рассчитана для {APRIL_MONTH}.')


def _norm_digits(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v))
    return s if s else None


def _flag_to_bool(v):
    if pd.isna(v):
        return False
    s = str(v).strip().lower()
    if s in {'1', 'true', 't', 'y', 'yes', 'да', 'д'}:
        return True
    if s in {'0', 'false', 'f', 'n', 'no', 'нет', ''}:
        return False
    try:
        return float(s) != 0.0
    except Exception:
        return False


def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


# 1) Только апрель + тариф Индивидуальный
work_df = final_df.copy()
if 'report_month' in work_df.columns:
    work_df = work_df[work_df['report_month'].astype(str) == april_label].copy()
else:
    print('WARN: в final_df нет report_month, используется весь final_df.')

work_df['tariff_name_norm'] = work_df.get('tariff_name', pd.Series(dtype=object)).astype(str).str.lower()
work_df['inn_norm'] = work_df.get('inn', pd.Series(dtype=object)).map(_norm_digits)
work_df['agr_id_key'] = work_df.get('agr_id', pd.Series(dtype=object)).astype(str).str.strip()
if 'r2_id' in work_df.columns:
    work_df['r2_id_key'] = work_df['r2_id'].astype(str).str.strip()
else:
    work_df['r2_id_key'] = ''

individual_apr_df = work_df[work_df['tariff_name_norm'].str.contains('индивиду', na=False)].copy()
print(f'Строк за апрель в final_df: {len(work_df):,}')
print(f'Строк с тарифом Индивидуальный: {len(individual_apr_df):,}')

# 2) Тянем budget/non_budget из companies по всем ИНН Индивидуального тарифа
inn_values = sorted([x for x in individual_apr_df['inn_norm'].dropna().astype(str).unique().tolist() if x])
companies_parts = []

for part in _chunks(inn_values, CHUNK_SIZE):
    inn_sql = ', '.join([f"'{x}'" for x in part])
    sql_companies_flags = f"""
    select
      regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_norm,
      cast(c.n_cmp as string) as n_cmp,
      cast(c.c_cmp_name as string) as company_name_comp,
      cast(c.budget_cmp as string) as budget_cmp,
      cast(c.non_budget as string) as non_budget
    from ods_alpha.scd1_companies c
    where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in ({inn_sql})
      and coalesce(c.ods_deleted_flg, '0') <> '1'
    """
    with imp:
        imp.execute('set MEM_LIMIT=8g')
        part_df = imp.fetch(sql_companies_flags)
    if part_df is not None and len(part_df):
        companies_parts.append(part_df)

if companies_parts:
    companies_flags_raw = pd.concat(companies_parts, ignore_index=True)
else:
    companies_flags_raw = pd.DataFrame(columns=['inn_norm', 'n_cmp', 'company_name_comp', 'budget_cmp', 'non_budget'])

if len(companies_flags_raw):
    companies_flags_raw['inn_norm'] = companies_flags_raw['inn_norm'].map(_norm_digits)
    companies_flags_raw['budget_bool'] = companies_flags_raw['budget_cmp'].map(_flag_to_bool)
    companies_flags_raw['non_budget_bool'] = companies_flags_raw['non_budget'].map(_flag_to_bool)

    companies_flags = (
        companies_flags_raw
        .groupby('inn_norm', as_index=False)
        .agg(
            budget_bool=('budget_bool', 'max'),
            non_budget_bool=('non_budget_bool', 'max'),
            n_cmp_cnt=('n_cmp', 'nunique'),
            company_name_comp=('company_name_comp', 'first')
        )
    )
else:
    companies_flags = pd.DataFrame(columns=['inn_norm', 'budget_bool', 'non_budget_bool', 'n_cmp_cnt', 'company_name_comp'])

# 3) Сегментация budget/non_budget
individual_apr_segmented = individual_apr_df.merge(companies_flags, on='inn_norm', how='left')
individual_apr_segmented['budget_bool'] = individual_apr_segmented['budget_bool'].fillna(False)
individual_apr_segmented['non_budget_bool'] = individual_apr_segmented['non_budget_bool'].fillna(False)


def _segment_row(r):
    b = bool(r.get('budget_bool', False))
    nb = bool(r.get('non_budget_bool', False))
    if b and not nb:
        return 'budget'
    if nb and not b:
        return 'non_budget'
    if b and nb:
        return 'conflict_both_flags'
    return 'unknown'


individual_apr_segmented['budget_segment'] = individual_apr_segmented.apply(_segment_row, axis=1)

# Готовые выборки
individual_budget_clients_apr = individual_apr_segmented[individual_apr_segmented['budget_segment'] == 'budget'].copy()
individual_non_budget_clients_apr = individual_apr_segmented[individual_apr_segmented['budget_segment'] == 'non_budget'].copy()
individual_conflict_clients_apr = individual_apr_segmented[individual_apr_segmented['budget_segment'] == 'conflict_both_flags'].copy()
individual_unknown_clients_apr = individual_apr_segmented[individual_apr_segmented['budget_segment'] == 'unknown'].copy()

print('\nРаспределение по сегментам (строки):')
print(individual_apr_segmented['budget_segment'].value_counts(dropna=False))

client_level = (
    individual_apr_segmented[['inn_norm', 'company_name', 'budget_segment']]
    .drop_duplicates()
)
print('\nРаспределение по сегментам (уникальные ИНН):')
print(client_level['budget_segment'].value_counts(dropna=False))

# 4) Точечная проверка TARGET_AGR_ID/TARGET_INN (agr_id из r2 слоя)
target_agr = str(TARGET_AGR_ID).strip()
target_inn = _norm_digits(TARGET_INN)

mask_target = (
    (individual_apr_segmented['inn_norm'] == target_inn)
    & (
        (individual_apr_segmented['agr_id_key'] == target_agr)
        | (individual_apr_segmented['r2_id_key'] == target_agr)
    )
)

target_apr_check = individual_apr_segmented[mask_target].copy()

print(f"\nTARGET check rows (ИНН={target_inn}, AGR={target_agr}): {len(target_apr_check):,}")

target_cols = [
    'report_month', 'inn', 'company_name', 'agr_id', 'r2_id', 'n_agr',
    'tariff_name', 'budget_bool', 'non_budget_bool', 'budget_segment',
    'company_name_comp', 'n_cmp_cnt'
]
target_cols = [c for c in target_cols if c in target_apr_check.columns]

display(target_apr_check[target_cols])

if target_apr_check.empty:
    print('TARGET не найден среди Индивидуального тарифа за апрель. Диагностика по final_df:')
    diag = work_df[(work_df['agr_id_key'] == target_agr) | (work_df['r2_id_key'] == target_agr)].copy()
    diag_cols = [c for c in ['report_month', 'inn', 'company_name', 'agr_id', 'r2_id', 'n_agr', 'tariff_name'] if c in diag.columns]
    display(diag[diag_cols])

# 5) Доп. подтверждение, что TARGET_AGR_ID присутствует в R2
sql_target_r2 = f"""
select
  cast(m.id as string) as r2_id,
  cast(m.c_cl_org as string) as cft_id,
  cast(m.c_tariff_plan as string) as c_tariff_plan,
  cast(tp.c_name as string) as tariff_name_r2
from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
where cast(m.id as string) = '{target_agr}'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    target_r2_info = imp.fetch(sql_target_r2)

if target_r2_info is None:
    target_r2_info = pd.DataFrame()

print(f'R2 target rows: {len(target_r2_info):,}')
display(target_r2_info)


## Секция 11. Non-budget (unknown включен): дельта Excel vs Озеро по комиссиям
Кратко: считаем `unknown` как `non_budget`, собираем отдельный `DataFrame` клиентов non-budget и считаем разницу `Excel - Lake` по всем комиссионным метрикам на базе уже рассчитанных `individual_apr_segmented` и `cmp`.

In [ ]:
import re
import pandas as pd

if 'individual_apr_segmented' not in globals() or individual_apr_segmented is None or len(individual_apr_segmented) == 0:
    raise RuntimeError('Сначала выполните Секцию 10 (нужен DataFrame individual_apr_segmented).')
if 'cmp' not in globals() or cmp is None or len(cmp) == 0:
    raise RuntimeError('Сначала выполните секцию сверки с Excel (нужен DataFrame cmp).')


def _norm_inn_local(v):
    if 'normalize_inn_q1' in globals():
        return normalize_inn_q1(v)
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v).strip())
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def _norm_agr_local(v):
    if 'normalize_agr_q1' in globals():
        return normalize_agr_q1(v)
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\s+', '', s)
    return s if s else None


# 1) unknown считаем как non_budget
nb_base = individual_apr_segmented.copy()
nb_base['budget_segment_eff'] = nb_base.get('budget_segment', pd.Series(dtype=object)).astype(str)
nb_base['budget_segment_eff'] = nb_base['budget_segment_eff'].replace({'unknown': 'non_budget'})

non_budget_all_df = nb_base[nb_base['budget_segment_eff'] == 'non_budget'].copy()

# Ключи для сшивки с cmp
if 'inn_key' in non_budget_all_df.columns:
    non_budget_all_df['inn_key'] = non_budget_all_df['inn_key'].map(_norm_inn_local)
else:
    non_budget_all_df['inn_key'] = non_budget_all_df.get('inn', pd.Series(dtype=object)).map(_norm_inn_local)

if 'agr_id_key' in non_budget_all_df.columns:
    non_budget_all_df['agr_id_key'] = non_budget_all_df['agr_id_key'].map(_norm_agr_local)
else:
    non_budget_all_df['agr_id_key'] = non_budget_all_df.get('agr_id', pd.Series(dtype=object)).map(_norm_agr_local)

key_cols = ['inn_key', 'agr_id_key']
meta_cols = [c for c in ['company_name', 'inn', 'tariff_name', 'budget_segment', 'budget_segment_eff'] if c in non_budget_all_df.columns]

non_budget_keys_df = (
    non_budget_all_df[key_cols + meta_cols]
    .dropna(subset=key_cols)
    .drop_duplicates(subset=key_cols)
    .reset_index(drop=True)
)

print(f'Non-budget (с unknown) ключей inn+agr_id: {len(non_budget_keys_df):,}')

# 2) Срез cmp только для non-budget
non_budget_comm_diff_df = cmp.merge(non_budget_keys_df, on=['inn_key', 'agr_id_key'], how='inner')
print(f'Строк в cmp по non-budget (unknown включен): {len(non_budget_comm_diff_df):,}')

# 3) Excel - Lake по комиссионным метрикам
commission_metrics = [
    'commission_from_ops',   # Комиссия (% с операций)
    'commission_monthly',    # Комиссия (₽ в месяц)
    'commission_total',      # Комиссия эквайринга
    'int_component',         # Комиссия МПС (IRF, ₽)
    'chod',                  # ЧОД
]
metric_ru = {
    'commission_from_ops': 'Комиссия (% с операций)',
    'commission_monthly': 'Комиссия (₽ в месяц)',
    'commission_total': 'Комиссия эквайринга',
    'int_component': 'Комиссия МПС (IRF, ₽)',
    'chod': 'ЧОД',
}

for m in commission_metrics:
    lake_col = f'{m}_lake'
    excel_col = f'{m}_excel'
    if lake_col not in non_budget_comm_diff_df.columns or excel_col not in non_budget_comm_diff_df.columns:
        print(f'WARN: нет колонок {lake_col}/{excel_col}; метрика {m} пропущена.')
        continue

    non_budget_comm_diff_df[lake_col] = pd.to_numeric(non_budget_comm_diff_df[lake_col], errors='coerce').fillna(0.0)
    non_budget_comm_diff_df[excel_col] = pd.to_numeric(non_budget_comm_diff_df[excel_col], errors='coerce').fillna(0.0)

    non_budget_comm_diff_df[f'{m}_delta_excel_minus_lake'] = (
        non_budget_comm_diff_df[excel_col] - non_budget_comm_diff_df[lake_col]
    )
    non_budget_comm_diff_df[f'{m}_abs_excel_minus_lake'] = (
        non_budget_comm_diff_df[f'{m}_delta_excel_minus_lake'].abs()
    )

# 4) Сводка
summary_rows = []
for m in commission_metrics:
    lake_col = f'{m}_lake'
    excel_col = f'{m}_excel'
    dcol = f'{m}_delta_excel_minus_lake'
    acol = f'{m}_abs_excel_minus_lake'

    if dcol not in non_budget_comm_diff_df.columns:
        continue

    summary_rows.append({
        'metric_key': m,
        'metric_name': metric_ru.get(m, m),
        'rows': int(len(non_budget_comm_diff_df)),
        'lake_total': float(non_budget_comm_diff_df[lake_col].sum()),
        'excel_total': float(non_budget_comm_diff_df[excel_col].sum()),
        'delta_total_excel_minus_lake': float(non_budget_comm_diff_df[dcol].sum()),
        'exact_match_rate_pct_tol_0_01': round((non_budget_comm_diff_df[acol] <= 0.01).mean() * 100, 2) if len(non_budget_comm_diff_df) else 0.0,
    })

non_budget_comm_diff_summary_df = pd.DataFrame(summary_rows)

# 5) Удобный вывод
detail_cols = ['agr_id_key', 'inn_key'] + meta_cols
for m in commission_metrics:
    for c in [f'{m}_lake', f'{m}_excel', f'{m}_delta_excel_minus_lake']:
        if c in non_budget_comm_diff_df.columns:
            detail_cols.append(c)

detail_cols = list(dict.fromkeys([c for c in detail_cols if c in non_budget_comm_diff_df.columns]))

print('\nСводка Excel - Lake по non-budget (unknown включен):')
display(non_budget_comm_diff_summary_df)

print('\nДетализация по non-budget (первые 50 строк):')
display(non_budget_comm_diff_df[detail_cols].head(50))

## Секция 12. Расходящиеся agr_id по commission_monthly (список с операциями)
Кратко: на базе результатов Секции 11 считаем количество уникальных `agr_id`, где есть расхождение `commission_monthly` между Excel и Озером, и выводим список с `trx_cnt` и `trx_sum`.

In [ ]:
# База: non-budget (unknown включен) из Секции 11
if 'non_budget_comm_diff_df' not in globals() or non_budget_comm_diff_df is None or len(non_budget_comm_diff_df) == 0:
    raise RuntimeError('Сначала выполните Секцию 11 (нужен DataFrame non_budget_comm_diff_df).')

source_df = non_budget_comm_diff_df.copy()

# Допуск для денежных расхождений (чтобы отсечь микрошум округления)
comm_monthly_tol = 0.01

required_cols = ['agr_id_key', 'commission_monthly_lake', 'commission_monthly_excel']
missing_required = [c for c in required_cols if c not in source_df.columns]
if missing_required:
    raise RuntimeError(f'В non_budget_comm_diff_df не хватает колонок: {missing_required}')

agg_map = {
    'commission_monthly_lake': 'sum',
    'commission_monthly_excel': 'sum',
}

# Добавляем в итог операции и суммы операций (если колонки присутствуют)
for c in ['trx_cnt_lake', 'trx_cnt_excel', 'trx_sum_lake', 'trx_sum_excel']:
    if c in source_df.columns:
        agg_map[c] = 'sum'

# Добавляем полезные атрибуты, если есть
for c in ['inn_key', 'company_name', 'inn', 'tariff_name']:
    if c in source_df.columns:
        agg_map[c] = 'first'

comm_by_agr = (
    source_df
    .groupby('agr_id_key', as_index=False)
    .agg(agg_map)
)

for c in ['commission_monthly_lake', 'commission_monthly_excel', 'trx_cnt_lake', 'trx_cnt_excel', 'trx_sum_lake', 'trx_sum_excel']:
    if c in comm_by_agr.columns:
        comm_by_agr[c] = pd.to_numeric(comm_by_agr[c], errors='coerce').fillna(0.0)

comm_by_agr['commission_monthly_delta_excel_minus_lake'] = (
    comm_by_agr['commission_monthly_excel'] - comm_by_agr['commission_monthly_lake']
)
comm_by_agr['commission_monthly_abs_delta'] = comm_by_agr['commission_monthly_delta_excel_minus_lake'].abs()

# Фильтр расходящихся agr_id
comm_monthly_mismatch_agr_df = comm_by_agr[
    comm_by_agr['commission_monthly_abs_delta'] > comm_monthly_tol
].copy()

comm_monthly_mismatch_agr_df = comm_monthly_mismatch_agr_df.sort_values(
    'commission_monthly_abs_delta', ascending=False
).reset_index(drop=True)

mismatch_agr_count = comm_monthly_mismatch_agr_df['agr_id_key'].nunique()
print(f"Количество agr_id с расхождением по commission_monthly (|delta| > {comm_monthly_tol}): {mismatch_agr_count:,}")

out_cols = [
    'agr_id_key',
    'inn_key', 'company_name', 'inn', 'tariff_name',
    'commission_monthly_lake', 'commission_monthly_excel', 'commission_monthly_delta_excel_minus_lake',
    'trx_cnt_lake', 'trx_cnt_excel', 'trx_sum_lake', 'trx_sum_excel'
]
out_cols = [c for c in out_cols if c in comm_monthly_mismatch_agr_df.columns]

print('Список agr_id с расхождением по commission_monthly:')
display(comm_monthly_mismatch_agr_df[out_cols])

## Секция 13. n_prc и витрина расхождений commission_monthly
Кратко: берем только `agr_id` с расхождениями из Секции 12, подтягиваем `n_prc` из `ods_alpha.scd1_agr_terms` (последняя активная `cf_ter_type='P'` запись за месяц) и выводим целевой набор колонок: `agr_id`, `n_agr`, `trx_summ_lake`, `trx_summ_excel`, `comission_monthly_lake`, `commission_monthly_excel`, `n_prc`, `comission_monthly_prc`, `excel_prc_delta`.

In [ ]:
import re
import pandas as pd

if 'comm_monthly_mismatch_agr_df' not in globals() or comm_monthly_mismatch_agr_df is None or len(comm_monthly_mismatch_agr_df) == 0:
    raise RuntimeError('Сначала выполните Секцию 12 (нужен DataFrame comm_monthly_mismatch_agr_df).')
if 'final_df' not in globals() or final_df is None or len(final_df) == 0:
    raise RuntimeError('Сначала сформируйте final_df.')
if 'imp' not in globals():
    raise RuntimeError('Нет подключения imp. Сначала выполните ячейку с connect().')


def _norm_agr_local(v):
    if 'normalize_agr_q1' in globals():
        return normalize_agr_q1(v)
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\s+', '', s)
    return s if s else None


def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


# 1) Берем только agr_id с расхождениями из Секции 12
section13_df = comm_monthly_mismatch_agr_df.copy()
section13_df['agr_id_key'] = section13_df['agr_id_key'].map(_norm_agr_local)

# Подстраховка: если в df есть колонка абсолютной дельты, отфильтруем реальные расхождения
if 'commission_monthly_abs_delta' in section13_df.columns:
    section13_df['commission_monthly_abs_delta'] = pd.to_numeric(section13_df['commission_monthly_abs_delta'], errors='coerce').fillna(0.0)
    section13_df = section13_df[section13_df['commission_monthly_abs_delta'] > 0.01].copy()

required_cols = ['agr_id_key', 'trx_sum_lake', 'commission_monthly_lake', 'commission_monthly_excel']
missing_required = [c for c in required_cols if c not in section13_df.columns]
if missing_required:
    raise RuntimeError(f'В comm_monthly_mismatch_agr_df не хватает колонок: {missing_required}')

# 2) Период для отбора записи в agr_terms
if 'month_start' in globals() and 'month_end' in globals():
    period_start = str(month_start)
    period_end = str(month_end)
elif 'report_month' in globals():
    report_ts = pd.to_datetime(str(report_month))
    period_start = report_ts.strftime('%Y-%m-%d')
    period_end = (report_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
else:
    report_ts = pd.to_datetime('2026-04-01')
    period_start = report_ts.strftime('%Y-%m-%d')
    period_end = (report_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')

period_label = pd.to_datetime(period_start).strftime('%Y-%m')
print(f'Период Секции 13: {period_start} .. {period_end}')

# 3) agr_id -> n_agr через final_df
final_map = final_df.copy()
if 'report_month' in final_map.columns:
    final_map = final_map[final_map['report_month'].astype(str) == period_label].copy()

if 'n_agr' not in final_map.columns:
    raise RuntimeError('В final_df нет n_agr, нельзя восстановить связь agr_id -> n_agr.')

final_map['agr_id_key'] = final_map['agr_id'].map(_norm_agr_local)
final_map['n_agr'] = final_map['n_agr'].astype(str).str.strip()
final_map = final_map[final_map['n_agr'] != ''].copy()

if 'd_valid_from' in final_map.columns:
    final_map['_d_valid_from_sort'] = pd.to_datetime(final_map['d_valid_from'], errors='coerce')
    final_map = final_map.sort_values(['agr_id_key', '_d_valid_from_sort'], ascending=[True, False])

agr_to_nagr = (
    final_map[['agr_id_key', 'n_agr']]
    .dropna(subset=['agr_id_key', 'n_agr'])
    .drop_duplicates(subset=['agr_id_key'], keep='first')
)

section13_df = section13_df.merge(agr_to_nagr, on='agr_id_key', how='left')

print(f'agr_id c расхождениями: {len(section13_df):,}')
print(f'agr_id, сматченных к n_agr: {int(section13_df["n_agr"].notna().sum()):,}')

# 4) Подтягиваем n_prc из agr_terms (последняя активная P-запись за период)
n_agr_values = sorted([x for x in section13_df['n_agr'].dropna().astype(str).unique().tolist() if x])
terms_parts = []

for part in _chunks(n_agr_values, 1000):
    n_agr_sql = ', '.join([f"'{x}'" for x in part])

    sql_terms_nprc = f"""
    with ranked as (
      select
        cast(t.n_agr as string) as n_agr,
        cast(t.n_prc as double) as n_prc,
        cast(t.d_valid_from as date) as d_valid_from,
        cast(t.d_valid_to as date) as d_valid_to,
        row_number() over (
          partition by cast(t.n_agr as string)
          order by cast(t.d_valid_from as date) desc,
                   coalesce(cast(t.d_valid_to as date), cast('2999-12-31' as date)) desc
        ) as rn
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) in ({n_agr_sql})
        and cast(t.d_valid_from as date) <= cast('{period_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{period_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    )
    select n_agr, n_prc
    from ranked
    where rn = 1
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        terms_part = imp.fetch(sql_terms_nprc)

    if terms_part is not None and len(terms_part):
        terms_parts.append(terms_part)

if terms_parts:
    nprc_df = pd.concat(terms_parts, ignore_index=True)
    nprc_df['n_agr'] = nprc_df['n_agr'].astype(str)
    nprc_df = nprc_df.drop_duplicates(subset=['n_agr'], keep='first')
else:
    nprc_df = pd.DataFrame(columns=['n_agr', 'n_prc'])

section13_df = section13_df.merge(nprc_df, on='n_agr', how='left')

# 5) Расчеты по запросу
section13_df['n_prc'] = pd.to_numeric(section13_df['n_prc'], errors='coerce')
section13_df['trx_summ_lake'] = pd.to_numeric(section13_df['trx_sum_lake'], errors='coerce')
section13_df['trx_summ_excel'] = pd.to_numeric(section13_df['trx_sum_excel'], errors='coerce') if 'trx_sum_excel' in section13_df.columns else pd.NA
section13_df['comission_monthly_lake'] = pd.to_numeric(section13_df['commission_monthly_lake'], errors='coerce')
section13_df['commission_monthly_excel'] = pd.to_numeric(section13_df['commission_monthly_excel'], errors='coerce')

# n_prc хранится как процент (например 2.00 = 2%), поэтому делим на 100
section13_df['comission_monthly_prc'] = section13_df['trx_summ_lake'] * (section13_df['n_prc'] / 100.0)
section13_df['excel_prc_delta'] = section13_df['commission_monthly_excel'] - section13_df['comission_monthly_prc']

section13_output_df = (
    section13_df
    .rename(columns={'agr_id_key': 'agr_id'})
    [[
        'agr_id', 'n_agr',
        'trx_summ_lake', 'trx_summ_excel',
        'comission_monthly_lake', 'commission_monthly_excel',
        'n_prc', 'comission_monthly_prc', 'excel_prc_delta'
    ]]
    .sort_values('excel_prc_delta', key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

print(f'Строк без n_prc: {int(section13_output_df["n_prc"].isna().sum()):,}')
print('Секция 13: список agr_id с расхождениями и расчетом по n_prc:')
display(section13_output_df)

# Совместимость с предыдущим именованием
section13_nprc_df = section13_output_df.copy()